# Colab environment setup for Lab Day 1
This is an environment launcher, not the final training notebook. Read `COLAB_WALKTHROUGH.md` for the implementation and submission steps.

Select Runtime > Change runtime type > a GPU. Set your student ID below. When asked to upload a file, choose `code/colab_bootstrap.py` from your local checkout.


In [ ]:
STUDENT_ID = "YOUR_STUDENT_ID"

from google.colab import files
from pathlib import Path
import importlib.util
import shutil

uploaded = files.upload()
if "colab_bootstrap.py" not in uploaded:
    raise ValueError("Upload code/colab_bootstrap.py from your local checkout.")
bootstrap_path = Path("/content/colab_bootstrap.py")
bootstrap_path.write_bytes(uploaded["colab_bootstrap.py"])
spec = importlib.util.spec_from_file_location("lab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
REPO_ROOT, SUBMISSION = bootstrap.setup(STUDENT_ID)
shutil.copy2(bootstrap_path, SUBMISSION / "code/colab_bootstrap.py")
print("Persistent submission:", SUBMISSION.resolve())
print("Open code/lab.ipynb from this folder in Google Drive.")


In [ ]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("Choose a GPU runtime before training; CPU is available for debugging.")


## Continue in your persistent lab notebook
Open the copied `lab.ipynb` in Google Drive. Replace its first code cell with the environment cell printed below. The printed cell includes its setup function and does not import the local helper. Exclude the setup helper and launcher from your final submission. Keep Parts 0?4 and complete the exercise modules.

The replacement cell supports Colab and a local run from the submission's `code/` directory. On Colab it mounts Drive before importing your persistent modules. On a new runtime it clones the course repository and regenerates processed data. It does not replace your completed modules.


In [ ]:
environment_cell = '"""Prepare a persistent Colab workspace without replacing student work."""\nfrom pathlib import Path\nimport importlib.util\nimport re\nimport shutil\nimport subprocess\nimport sys\n\n\ndef setup(student_id: str) -> tuple[Path, Path]:\n    """Return the repository root and submission directory."""\n    if not re.fullmatch(r"[A-Za-z0-9_-]+", student_id) or student_id == "YOUR_STUDENT_ID":\n        raise ValueError("Replace YOUR_STUDENT_ID with your actual student ID.")\n    root = Path("/content/K4-Track4-Day1-Neuralnetwork")\n    if not root.exists():\n        subprocess.run([\n            "git", "clone", "https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git",\n            str(root),\n        ], check=True)\n    for name in ("numpy", "pandas", "sklearn", "matplotlib", "openpyxl", "torch"):\n        if importlib.util.find_spec(name) is None:\n            package = "scikit-learn" if name == "sklearn" else name\n            subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)\n    from google.colab import drive\n    drive.mount("/content/drive")\n    saved = Path("/content/drive/MyDrive/NeuralNetworkLab") / f"submission_{student_id}"\n    (saved / "code").mkdir(parents=True, exist_ok=True)\n    for source in (root / "code").iterdir():\n        target = saved / "code" / source.name\n        if source.is_file() and not target.exists():\n            shutil.copy2(source, target)\n    for source, target in (\n        (root / "templates/REPORT_TEMPLATE.md", saved / "REPORT.md"),\n        (root / "templates/experiment_table_template.xlsx", saved / "experiments.xlsx"),\n        (root / "templates/experiment_table_template.xlsx", saved / "code/experiment_table_template.xlsx"),\n    ):\n        if not target.exists():\n            shutil.copy2(source, target)\n    for name in ("figures", "results"):\n        (saved / name).mkdir(exist_ok=True)\n    submission = root / saved.name\n    if not submission.exists():\n        submission.symlink_to(saved, target_is_directory=True)\n    elif submission.resolve() != saved.resolve():\n        raise RuntimeError(f"Another workspace already exists at {submission}.")\n    processed = root / "data/processed"\n    if not all((processed / name).exists() for name in ("train.npz", "eval.npz")):\n        subprocess.run([sys.executable, "scripts/split_data.py"], cwd=root, check=True)\n    return root, submission\n\n\nimport os\nimport sys\nimport json\nimport time\nimport subprocess\nfrom pathlib import Path\n\nSTUDENT_ID = "YOUR_STUDENT_ID"\nif "google.colab" in sys.modules:\n    from google.colab import drive\n    drive.mount("/content/drive")\n    persistent_code = Path("/content/drive/MyDrive/NeuralNetworkLab") / f"submission_{STUDENT_ID}" / "code"\n    sys.path.insert(0, str(persistent_code))\n    REPO_ROOT, submission = setup(STUDENT_ID)\n    os.chdir(submission / "code")\nelse:\n    REPO_ROOT = Path.cwd().resolve().parents[1]\n    submission = Path.cwd().resolve().parent\n\nsys.path.insert(0, str(Path.cwd()))\nOUT_DIR = str(submission)\nREPO_ROOT = str(REPO_ROOT)\nimport numpy as np\nimport torch\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nprint("PyTorch:", torch.__version__, "Device:", device)\nif device == "cuda":\n    print("GPU:", torch.cuda.get_device_name(0))\nfor name in ("figures", "results"):\n    Path(OUT_DIR, name).mkdir(parents=True, exist_ok=True)\n\nfrom data import prepare_data\nfrom model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats\nfrom optimizer import build_optimizer, clip_gradients\nfrom train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval\nfrom plots import plot_run, plot_compare\nfrom results_table import save_result, load_results, to_row, write_xlsx\n'
print(environment_cell.replace('STUDENT_ID = "YOUR_STUDENT_ID"', f"STUDENT_ID = {STUDENT_ID!r}"))


## Before training
Complete the exercise functions before running the lab. Use the workbook template at `Path.cwd() / "experiment_table_template.xlsx"` in Part 4, because it is included inside your submission.

Do not use eval to choose settings. Start with the required SGD+momentum baseline; run the health checks, tune learning rate using validation, and repeat the selected baseline across seeds. See the walkthrough for the experiment plan.

## Before packaging
Save the executed lab notebook with outputs into the persistent submission's `code/lab.ipynb`. Complete the report and workbook and run the official evaluator. Exclude data, weights and cache files. This launcher is not a substitute for the executed lab notebook.
